# Artefact: the result of reading

The `Artefact` is the hand-over between the [reader](reader.ipynb) and everything after it. It holds the entities, the RDF graph, the IRI of the documented artefact, the imported modules, the warnings and the format. Viewers, the builder and the API depend on this object only: they never need the reader that built it.

This notebook inspects an artefact, read from a small ontology.

:::{warning} Beta version · unstable
LODE 2 is under development and in beta: anything described in this documentation may change. This page describes internals that can change without notice.
:::

## Methods to inspect an artefact

| Method | What it gives |
|---|---|
| `OwlOntology(source)`, `lode.read(source, read_as)` | Read a source. Both return an `Artefact`. |
| `artefact.read_as` | The name of the format the artefact was read as. |
| `artefact.get_entities(cls=None)` | The entities, as a list; with a model class, only those of that class and its subclasses. |
| `artefact.get_entities_by_iri(iri)` | The entities registered under an IRI, or `None`. |
| `artefact.entities` | The dictionary behind the two methods above: IRI to set of entities. It also holds the literals. |
| `artefact.artefact_iri`, `artefact.artefact_base_iri` | The IRI of the documented ontology or vocabulary, and its namespace. |
| `artefact.graph`, `artefact.main_graph()` | The RDF graph, and only the triples of the loaded artefact. |
| `artefact.imported_uris` | The IRIs of the modules pulled in by imports. |
| `artefact.warnings` | The reader warnings, when requested. |
| `artefact.provenance(entity)` | The triples an entity was built from. |
| `artefact.profile`, `artefact.reader` | The profile of the format, and the reader that built the artefact. |

## Examples

One for each method of the table, on a small ontology.

### `OwlOntology(source)` and `lode.read(source, read_as)`

In [1]:
import tempfile
from pathlib import Path

from lode import OwlOntology

TURTLE = """
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix ex: <http://example.org/zoo#> .

<http://example.org/zoo> a owl:Ontology ; rdfs:label "Zoo"@en .
ex:Animal a owl:Class ; rdfs:label "Animal"@en .
ex:Cat a owl:Class ; rdfs:label "Cat"@en ; rdfs:subClassOf ex:Animal .
"""

path = Path(tempfile.mkdtemp()) / "zoo.ttl"
path.write_text(TURTLE, encoding="utf-8")

import lode

artefact = OwlOntology(str(path))
same = lode.read(str(path), "owl")

print(type(artefact).__name__, type(same).__name__)

OwlOntology OwlOntology


### `artefact.read_as`

In [2]:
print(artefact.read_as)

owl


### `artefact.get_entities(cls=None)`

Literals are values, not entities, and are never listed.

In [3]:
from collections import Counter

from lode.models import Concept

print(dict(Counter(type(e).__name__ for e in artefact.get_entities())))
print(artefact.get_entities(Concept))

{'Concept': 3, 'Model': 1, 'Datatype': 1}
[Concept('http://example.org/zoo#Cat'), Concept('http://example.org/zoo#Animal'), Concept('http://www.w3.org/2002/07/owl#Thing'), Datatype('http://www.w3.org/2000/01/rdf-schema#Literal')]


### `artefact.get_entities_by_iri(iri)`

A set, because one IRI can be used for more than one kind of entity (punning).

In [4]:
artefact.get_entities_by_iri("http://example.org/zoo#Cat")

{Concept('http://example.org/zoo#Cat')}

### `artefact.entities`

The dictionary behind `get_entities`. Treat it as read-only: the reader shares it with the artefact.

In [5]:
print(len(artefact.entities), "keys")
for key in sorted(map(str, artefact.entities))[:5]:
    print(key)

8 keys
LITERAL::Animal@en^^
LITERAL::Cat@en^^
LITERAL::Zoo@en^^
http://example.org/zoo
http://example.org/zoo#Animal


### `artefact.artefact_iri` and `artefact.artefact_base_iri`

In [6]:
print(artefact.artefact_iri)
print(artefact.artefact_base_iri)

http://example.org/zoo
http://example.org/


### `artefact.graph` and `artefact.main_graph()`

They are the same graph here. With imports requested, `graph` also holds the imported modules and `main_graph()` keeps only the triples of the loaded artefact.

In [7]:
print(len(artefact.graph), len(artefact.main_graph()))

7 7


### `artefact.imported_uris`

In [8]:
print(artefact.imported_uris)

set()


### `artefact.warnings`

Empty unless the artefact is read with `warnings=True`.

In [9]:
checked = OwlOntology(str(path), warnings=True)
print(artefact.warnings, checked.warnings)

[] []


### `artefact.provenance(entity)`

Computed only when asked for, because most entities are never inspected.

In [10]:
cat = next(iter(artefact.get_entities_by_iri("http://example.org/zoo#Cat")))
print(artefact.provenance(cat).serialize(format="turtle"))

@prefix ns1: <http://www.w3.org/2000/01/rdf-schema#> .

<http://example.org/zoo#Cat> a <http://www.w3.org/2002/07/owl#Class> ;
    ns1:label "Cat"@en ;
    ns1:subClassOf <http://example.org/zoo#Animal> .




### `artefact.profile` and `artefact.reader`

In [11]:
print(artefact.profile.name)
print(type(artefact.reader).__name__)

owl
OwlReader


## How LODE uses it

The reader fills the artefact once, and everything after it reads from there: the [viewer](../viewer.md), the static builder and the API. Reading is described in [the Python API](../usage/python_library.md).

## Writing your own artefact class

This page only inspects the artefact. The `Artefact` class of a format carries no code: a subclass that the [profile](profile.ipynb) names, so that users can read the format with `MyFormat(path)`. To write it, go to [Extending LODE, the artefact class](../extending.md#the-artefact-class).